In [2]:
import heapq

In [1]:
class NavigationStack:
    def __init__(self, max_size=10):
        self.items = []
        self.max_size = max_size

    def push(self, location):
        if len(self.items) >= self.max_size:
            self.items.pop(0)  # Keep last 10 visited
        self.items.append(location)

    def pop(self):
        return self.items.pop() if self.items else None

    def show_history(self):
        return list(reversed(self.items))  # Most recent first


In [3]:
class ShuttleQueue:
    def __init__(self):
        self.items = []

    def enqueue(self, trip):
        self.items.append(trip)

    def dequeue(self):
        return self.items.pop(0) if self.items else None

    def show_queue(self):
        return list(self.items)

In [4]:
class BSTNode:
    def __init__(self, room_code, details):
        self.room_code = room_code
        self.details = details
        self.left = None
        self.right = None

In [5]:
class RoomBST:
    def __init__(self):
        self.root = None

    def insert(self, room_code, details):
        self.root = self._insert_rec(self.root, room_code, details)

    def _insert_rec(self, node, room_code, details):
        if node is None:
            return BSTNode(room_code, details)
        if room_code < node.room_code:
            node.left = self._insert_rec(node.left, room_code, details)
        elif room_code > node.room_code:
            node.right = self._insert_rec(node.right, room_code, details)
        return node

    def search(self, room_code):
        curr = self.root
        while curr:
            if room_code == curr.room_code:
                return curr.details
            curr = curr.left if room_code < curr.room_code else curr.right
        return None

In [6]:
class RoomHashTable:
    def __init__(self, size=11):
        self.size = size
        self.table = [[] for _ in range(size)]

    def _hash(self, key):
        return sum(ord(ch) for ch in key) % self.size

    def insert(self, room_code, details):
        idx = self._hash(room_code)
        self.table[idx].append((room_code, details))

    def search(self, room_code):
        idx = self._hash(room_code)
        for k, v in self.table[idx]:
            if k == room_code:
                return v
        return None

In [7]:
class BusiestLocationsHeap:
    def __init__(self):
        self.heap = []

    def build(self, locations):
        self.heap = list(locations)
        for i in range(len(self.heap) // 2 - 1, -1, -1):
            self._sift_down(i)

    def _sift_down(self, idx):
        n = len(self.heap)
        while True:
            smallest = idx
            left, right = 2 * idx + 1, 2 * idx + 2
            if left < n and self.heap[left] < self.heap[smallest]:
                smallest = left
            if right < n and self.heap[right] < self.heap[smallest]:
                smallest = right
            if smallest != idx:
                self.heap[idx], self.heap[smallest] = self.heap[smallest], self.heap[idx]
                idx = smallest
            else:
                break

    def get_top_k(self, k=3):
        # Sort copy descending by footfall to display top k busiest
        ranked = sorted(self.heap, reverse=True)
        return ranked[:k]


In [8]:
class CampusGraph:
    def __init__(self, vertices):
        self.vertices = sorted(vertices)
        self.adj = {v: [] for v in self.vertices}

    def add_edge(self, u, v, weight):
        self.adj[u].append((v, weight))
        self.adj[v].append((u, weight))
        self.adj[u].sort()
        self.adj[v].sort()

    def dfs(self, start):
        visited = []
        stack = [start]
        while stack:
            node = stack.pop()
            if node not in visited:
                visited.append(node)
                for neighbour, _ in reversed(self.adj[node]):
                    if neighbour not in visited:
                        stack.append(neighbour)
        return visited

    def bfs(self, start):
        visited = [start]
        queue = [start]
        while queue:
            node = queue.pop(0)
            for neighbour, _ in self.adj[node]:
                if neighbour not in visited:
                    visited.append(neighbour)
                    queue.append(neighbour)
        return visited

    def dijkstra(self, start, end):
        distances = {v: float('inf') for v in self.vertices}
        previous = {v: None for v in self.vertices}
        distances[start] = 0
        pq = [(0, start)]

        while pq:
            curr_dist, curr_node = heapq.heappop(pq)
            if curr_node == end:
                break
            if curr_dist > distances[curr_node]:
                continue
            for neighbour, weight in self.adj[curr_node]:
                new_dist = curr_dist + weight
                if new_dist < distances[neighbour]:
                    distances[neighbour] = new_dist
                    previous[neighbour] = curr_node
                    heapq.heappush(pq, (new_dist, neighbour))

        path = []
        curr = end
        while curr is not None:
            path.insert(0, curr)
            curr = previous[curr]
        return path, distances[end]


In [ ]:
def main():
    buildings = ["GATE", "ADM", "CAF", "HST", "ICT", "LEC1", "LEC2", "LIB", "MED", "RAD", "SPT"]
    graph = CampusGraph(buildings)
    edges = [
        ("GATE", "ADM", 2), ("GATE", "LIB", 4), ("ADM", "LIB", 3),
        ("ADM", "HST", 5), ("ADM", "RAD", 4), ("LIB", "ICT", 2),
        ("LIB", "LEC1", 3), ("HST", "CAF", 4), ("HST", "SPT", 7),
        ("ICT", "RAD", 3), ("ICT", "LEC1", 3), ("LEC1", "LEC2", 2),
        ("LEC2", "CAF", 3), ("LEC2", "MED", 4), ("MED", "SPT", 3)
    ]
    for u, v, w in edges:
        graph.add_edge(u, v, w)

    room_bst = RoomBST()
    room_ht = RoomHashTable()
    rooms = [
        ("ICT-101", "Software Engineering Lab (ICT Building, Floor 1)"),
        ("LIB-202", "Postgraduate Reading Room (Library Annex, Floor 2)"),
        ("ADM-01", "Academic Registrar Office (Administration, Ground Floor)"),
        ("LEC1-A", "Main Computing Theatre (Lecture Hall 1)"),
        ("MED-03", "Outpatient Consultation Room (Sickbay)")
    ]
    for code, desc in rooms:
        room_bst.insert(code, desc)
        room_ht.insert(code, desc)

    footfall_heap = BusiestLocationsHeap()
    footfall_heap.build([
        (650, "CAF"), (520, "LIB"), (440, "LEC1"), (380, "LEC2"),
        (290, "ICT"), (240, "HST"), (150, "ADM"), (90, "SPT")
    ])

    history = NavigationStack()
    shuttle = ShuttleQueue()
    shuttle.enqueue("08:00 AM - Main Gate -> Hostels -> Town Centre")
    shuttle.enqueue("09:00 AM - Main Gate -> Hostels -> Town Centre")

    while True:
        print("\n=== MMU CAMPUS NAVIGATION & RESOURCE SYSTEM ===")
        print("1. Find Shortest Route (Dijkstra)")
        print("2. View Campus Traversals (DFS & BFS from GATE)")
        print("3. Lookup Room (BST & Hash Table)")
        print("4. View Top 3 Busiest Locations (Min-Heap)")
        print("5. View Navigation History (Stack)")
        print("6. Manage Shuttle Queue (FIFO Queue)")
        print("0. Exit")

        choice = input("Select an option: ").strip()

        if choice == "1":
            src = input("Enter start building (e.g., GATE): ").strip().upper()
            dst = input("Enter destination building (e.g., SPT): ").strip().upper()
            if src in graph.adj and dst in graph.adj:
                path, mins = graph.dijkstra(src, dst)
                history.push(f"{src} -> {dst} ({mins} mins)")
                print(f"Shortest Path: {' -> '.join(path)} | Total Time: {mins} mins")
            else:
                print("Invalid building code.")

        elif choice == "2":
            print("DFS from GATE:", " -> ".join(graph.dfs("GATE")))
            print("BFS from GATE:", " -> ".join(graph.bfs("GATE")))

        elif choice == "3":
            rc = input("Enter room code (e.g., ICT-101): ").strip().upper()
            bst_res = room_bst.search(rc)
            ht_res = room_ht.search(rc)
            if bst_res and ht_res:
                history.push(f"Searched Room {rc}")
                print(f"[BST Match]:        {bst_res}")
                print(f"[Hash Table Match]: {ht_res}")
            else:
                print("Room not found.")

        elif choice == "4":
            print("Top 3 Busiest Campus Locations:")
            for rank, (count, b_code) in enumerate(footfall_heap.get_top_k(3), 1):
                print(f"  {rank}. {b_code} ({count} daily visitors)")

        elif choice == "5":
            print("Recent Navigation History (Most Recent First):", history.show_history())

        elif choice == "6":
            print("Current Shuttle Queue:", shuttle.show_queue())
            act = input("Type 'A' to add shuttle, 'D' to dispatch next, or Enter to skip: ").strip().upper()
            if act == "A":
                trip = input("Enter shuttle schedule details: ").strip()
                shuttle.enqueue(trip)
                print("Added to queue.")
            elif act == "D":
                dispatched = shuttle.dequeue()
                print(f"Dispatched: {dispatched}" if dispatched else "Queue is empty.")
        elif choice == "0":
            print("Exiting system.")
            break
        else:
            print("Invalid option.")

In [ ]:
if __name__ == "__main__":
    main()